# Fine-tune Mini BERT for sentiment, and compare it with Google's

Run this **after** `pretrain_colab.ipynb` has finished: it uses the model saved in `MyDrive/mini-bert/run-mini/model`.

Two models of exactly the same size learn the same task with exactly the same recipe:
- **ours**, pre-trained from scratch in `pretrain_colab.ipynb`
- **Google's BERT-Mini**, pre-trained by Google on far more text

The task is SST-2: sentences from movie reviews, positive or negative. Each model is fine-tuned for 4 epochs at three learning rates with three seeds each, and scored on SST-2's validation set (872 sentences it never trains on).

**To run:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, and allow access to Google Drive when asked.

**If Colab disconnects**, Run all again: every finished run is saved to `finetune_results.json` and skipped next time.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Run all."
print(torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)

## 1. Drive and code

In [ ]:
import base64
import hashlib
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
from huggingface_hub import hf_hub_download

drive.mount("/content/drive", timeout_ms=600_000)  # 10 minutes to approve Drive access
HOME = Path("/content/drive/MyDrive/mini-bert")  # the code, the pre-trained model and the results live here
RUN = HOME / "run-mini"
CODE = Path("/content/mini-bert")
CODE_SHA256 = "0992ea04b89f0ee7d29630f25abe1a6f51d14806575a6e83a24bdfc6285e90c8"  # the code pretrain_colab.ipynb uses
EXTRA_SHA256 = "9062c9ae86392c247ebc31599c3a39b4eba7de31843bd961636939d3f4dade2e"  # plus fine-tuning: minibert/finetune.py and the new commands


def unpack(paths, expected, name):
    packed = base64.b64decode("".join(path.read_text().strip() for path in paths))
    if hashlib.sha256(packed).hexdigest() != expected:
        raise RuntimeError(f"The code files in {HOME} are damaged or out of date. Upload them again.")
    archive = Path("/content") / name
    archive.write_bytes(packed)
    shutil.unpack_archive(archive, CODE, format="xztar")


# Get the code: from the copy in Drive if there is one, else from GitHub.
parts = [HOME / f"code.part{n}.b64" for n in (1, 2, 3)]
extra = HOME / "code-finetune.b64"
shutil.rmtree(CODE, ignore_errors=True)
if all(path.exists() for path in [*parts, extra]):
    unpack(parts, CODE_SHA256, "mini-bert-code.tar.xz")
    unpack([extra], EXTRA_SHA256, "mini-bert-finetune.tar.xz")
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/omdkpatel810-ops/mini-bert", str(CODE)], check=True)

# Add Google's BERT vocabulary (downloaded rather than stored in Drive), then install.
vocab = CODE / "minibert" / "data" / "bert-base-uncased-vocab.txt"
vocab.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download("bert-base-uncased", "vocab.txt"), vocab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(CODE)], check=True)
importlib.invalidate_caches()

from minibert.finetune import sst2  # noqa: E402  (only importable once installed)

assert (RUN / "model" / "model.safetensors").exists(), "No pre-trained model yet: finish pretrain_colab.ipynb first."
print("Code ready.")

## 2. SST-2

In [ ]:
train, valid = sst2("train"), sst2("validation")
print(f"{len(train):,} training sentences, {len(valid):,} validation sentences. For example:")
for example in train[:3]:
    print(" ", ["negative", "positive"][example.label], "|", example.text)

## 3. Fine-tune both models

18 runs in all: 2 models × 3 learning rates × 3 seeds. Each line shows one finished run.

In [ ]:
import json

from minibert.checkpoint import load
from minibert.finetune import SST2_LABELS, FinetuneSettings, SentenceClassifier, finetune

MODELS = {"ours (Mini BERT)": RUN / "model", "Google BERT-Mini": "google/bert_uncased_L-4_H-256_A-4"}
RATES = [3e-4, 1e-4, 5e-5]
SEEDS = [1, 2, 3]
RESULTS = HOME / "finetune_results.json"
results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}  # runs finished before a disconnect

for name, source in MODELS.items():
    pretrained, tokenizer = load(source)
    for rate in RATES:
        for seed in SEEDS:
            key = f"{name} | lr {rate:g} | seed {seed}"
            if key in results:
                continue
            model = SentenceClassifier.from_pretrained(pretrained, SST2_LABELS, seed=seed)
            result = finetune(
                model,
                tokenizer,
                train,
                valid,
                FinetuneSettings(learning_rate=rate, seed=seed),
                report=lambda line: None,
            )
            results[key] = {
                "model": name,
                "learning_rate": rate,
                "seed": seed,
                **{field: result[field] for field in ("accuracy", "loss", "history", "seconds")},
            }
            RESULTS.write_text(json.dumps(results, indent=2))
            print(f"{key}: {result['accuracy']:.1%} right ({result['seconds'] / 60:.1f} min)")
print("All runs done.")

## 4. Results

In [ ]:
import statistics

import matplotlib.pyplot as plt

from minibert.finetune import classify, save_classifier

GOOGLE_PUBLISHED = 0.859  # Google's own BERT-Mini SST-2 score (GLUE test set, best of their own grid)
summary = {"gpu": torch.cuda.get_device_name(0), "google_published_test": GOOGLE_PUBLISHED, "models": {}}
for name in MODELS:
    by_rate = {rate: [results[f"{name} | lr {rate:g} | seed {seed}"]["accuracy"] for seed in SEEDS] for rate in RATES}
    best = max(RATES, key=lambda rate: statistics.mean(by_rate[rate]))
    scores = by_rate[best]
    summary["models"][name] = {
        "best_learning_rate": best,
        "mean": statistics.mean(scores),
        "stdev": statistics.stdev(scores),
        "seeds": scores,
        "by_learning_rate": {f"{rate:g}": statistics.mean(by_rate[rate]) for rate in RATES},
    }
    print(
        f"{name:<18} {statistics.mean(scores):.1%} ± {statistics.stdev(scores):.1%} "
        f"(learning rate {best:g}; seeds: {', '.join(f'{s:.1%}' for s in scores)})"
    )
print(f"{'Google, published':<18} {GOOGLE_PUBLISHED:.1%} (their test-set score, for reference)")
summary["minutes"] = round(sum(r["seconds"] for r in results.values()) / 60, 1)

# Fine-tune each model once more at its best learning rate, keep ours, and try some sentences.
examples = [
    "a gorgeous , witty and moving film .",
    "the plot was predictable and the jokes fell flat .",
    "not bad at all .",
    "i wanted to love it , but it was a mess .",
    "the acting is fine , the story is not .",
]
summary["examples"] = {}
for name, source in MODELS.items():
    pretrained, tokenizer = load(source)
    rate = summary["models"][name]["best_learning_rate"]
    model = SentenceClassifier.from_pretrained(pretrained, SST2_LABELS, seed=1)
    finetune(model, tokenizer, train, valid, FinetuneSettings(learning_rate=rate, seed=1), report=lambda line: None)
    predictions = classify(model, tokenizer, examples)
    summary["examples"][name] = [[p.label, round(p.probability, 3)] for p in predictions]
    if name.startswith("ours"):
        save_classifier(model.cpu(), tokenizer, HOME / "sentiment-ours")
print()
for i, text in enumerate(examples):
    guesses = "   ".join(f"{name.split(' ')[0]}: {summary['examples'][name][i][0]}" for name in MODELS)
    print(f"{text:<52} {guesses}")
(HOME / "finetune_summary.json").write_text(json.dumps(summary, indent=2))

figure, axis = plt.subplots(figsize=(7, 4.4))
width = 0.38
for offset, name in zip((-width / 2, width / 2), MODELS, strict=True):
    for i, rate in enumerate(RATES):
        scores = [100 * results[f"{name} | lr {rate:g} | seed {seed}"]["accuracy"] for seed in SEEDS]
        axis.bar(
            i + offset,
            statistics.mean(scores),
            width,
            color="C0" if name.startswith("ours") else "C1",
            label=name if i == 0 else None,
        )
        axis.scatter([i + offset] * len(scores), scores, color="black", s=10, zorder=3)
axis.axhline(100 * GOOGLE_PUBLISHED, linestyle="--", color="gray", label="Google's published BERT-Mini (test)")
axis.set_xticks(range(len(RATES)), [f"lr {rate:g}" for rate in RATES])
lowest = min(100 * r["accuracy"] for r in results.values())
axis.set_ylim(min(70, 5 * (lowest // 5) - 5), 95)
axis.set_ylabel("% of validation sentences right")
axis.set_title(f"SST-2 sentiment after fine-tuning (bars: mean of {len(SEEDS)} seeds; dots: each seed)")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=8, frameon=False)
figure.tight_layout()
figure.savefig(HOME / "finetune.png", dpi=150)
print(f"Saved finetune_summary.json, finetune.png and the sentiment-ours model to {HOME}.")